In [1]:
# %% [markdown]
"""
## Save Preprocessor Correctly for Web App
"""

# %%
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Create sample data to demonstrate (replace with your actual data loading)
# df = pd.read_excel('../data/raw/hcc-data-complete-balanced.xlsx')

# Define feature types
categorical_features = ['Gender', 'Symptoms', 'Encephalopathy', 'Ascites', 
                       'Alcohol', 'HBsAg', 'HCVAb', 'Cirrhosis', 'Diabetes', 'Smoking']

numerical_features = ['Age', 'PS', 'AFP', 'ALT', 'AST', 'Albumin', 'Total_Bil', 
                     'INR', 'Platelets', 'Creatinine', 'Nodules', 'Major_Dim']

# Create preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Create column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessor created successfully!")

# %%
# Fit the preprocessor with sample data (replace with your actual training data)
# For demonstration, creating a small sample dataset
sample_data = pd.DataFrame({
    'Age': [65, 45, 72], 
    'Gender': ['Male', 'Female', 'Male'],
    'Symptoms': ['Yes', 'No', 'Yes'],
    'PS': [0, 1, 0],
    'Encephalopathy': ['No', 'No', 'Yes'],
    'Ascites': ['No', 'Yes', 'No'],
    'AFP': [250, 45, 1200],
    'ALT': [45, 32, 78],
    'AST': [50, 40, 65],
    'Albumin': [3.8, 4.2, 3.2],
    'Total_Bil': [1.2, 0.8, 2.1],
    'INR': [1.1, 1.0, 1.3],
    'Platelets': [150, 220, 90],
    'Creatinine': [0.9, 0.8, 1.2],
    'Nodules': [2, 1, 3],
    'Major_Dim': [5.0, 3.2, 7.1],
    'Alcohol': ['Yes', 'No', 'Yes'],
    'HBsAg': ['No', 'No', 'Yes'],
    'HCVAb': ['Yes', 'No', 'No'],
    'Cirrhosis': ['Yes', 'No', 'Yes'],
    'Diabetes': ['No', 'Yes', 'No'],
    'Smoking': ['Yes', 'No', 'Yes']
})

# Fit the preprocessor
preprocessor.fit(sample_data[numerical_features + categorical_features])
print("Preprocessor fitted successfully!")

# %%
# Get feature names after preprocessing
feature_names = numerical_features.copy()

# Add one-hot encoded feature names
categorical_processor = preprocessor.named_transformers_['cat']
onehot_encoder = categorical_processor.named_steps['onehot']

categorical_features_encoded = onehot_encoder.get_feature_names_out(categorical_features)
feature_names.extend(categorical_features_encoded)

print(f"Total features after encoding: {len(feature_names)}")
print("Feature breakdown:")
print(f"  Numerical: {len(numerical_features)}")
print(f"  Encoded categorical: {len(categorical_features_encoded)}")
print("\nFirst 10 features:", feature_names[:10])
print("Last 10 features:", feature_names[-10:])

# %%
# Save the preprocessor with all necessary information
preprocessor_data = {
    'preprocessor': preprocessor,
    'feature_names': feature_names,
    'original_features': numerical_features + categorical_features,
    'categorical_features': categorical_features,
    'numerical_features': numerical_features,
    'feature_categories': {
        'demographic': ['Age', 'Gender'],
        'clinical': ['Symptoms', 'PS', 'Encephalopathy', 'Ascites'],
        'laboratory': ['AFP', 'ALT', 'AST', 'Albumin', 'Total_Bil', 'INR', 'Platelets', 'Creatinine'],
        'tumor_characteristics': ['Nodules', 'Major_Dim'],
        'comorbidities': ['Alcohol', 'HBsAg', 'HCVAb', 'Cirrhosis', 'Diabetes', 'Smoking']
    }
}

# Create directory if it doesn't exist
os.makedirs('../models/preprocessing', exist_ok=True)

# Save the preprocessor
joblib.dump(preprocessor_data, '../models/preprocessing/preprocessor.pkl')

print("✅ Preprocessor saved successfully!")
print(f"📁 Location: ../models/preprocessing/preprocessor.pkl")
print(f"📊 Total features: {len(feature_names)}")
print(f"🔧 Preprocessor type: {type(preprocessor)}")

# %%
# Verify the saved preprocessor
print("\n🔍 Verifying saved preprocessor...")
loaded_data = joblib.load('../models/preprocessing/preprocessor.pkl')

print(f"Type: {type(loaded_data)}")
if isinstance(loaded_data, dict):
    print(f"Keys: {list(loaded_data.keys())}")
    print(f"Feature names: {len(loaded_data['feature_names'])}")
    print(f"Preprocessor available: {'preprocessor' in loaded_data}")
    
    # Test transformation with sample data
    if 'preprocessor' in loaded_data:
        sample_test_data = pd.DataFrame({
            'Age': [65], 'Gender': ['Male'], 'Symptoms': ['Yes'], 'PS': [0],
            'Encephalopathy': ['No'], 'Ascites': ['No'], 'AFP': [250],
            'ALT': [45], 'AST': [50], 'Albumin': [3.8], 'Total_Bil': [1.2],
            'INR': [1.1], 'Platelets': [150], 'Creatinine': [0.9],
            'Nodules': [2], 'Major_Dim': [5.0], 'Alcohol': ['Yes'],
            'HBsAg': ['No'], 'HCVAb': ['Yes'], 'Cirrhosis': ['Yes'],
            'Diabetes': ['No'], 'Smoking': ['Yes']
        })
        
        try:
            transformed = loaded_data['preprocessor'].transform(sample_test_data)
            print(f"✅ Transformation successful!")
            print(f"   Input shape: {sample_test_data.shape}")
            print(f"   Output shape: {transformed.shape}")
            print(f"   Expected features: {len(loaded_data['feature_names'])}")
        except Exception as e:
            print(f"❌ Transformation failed: {e}")

Preprocessor created successfully!
Preprocessor fitted successfully!
Total features after encoding: 22
Feature breakdown:
  Numerical: 12
  Encoded categorical: 10

First 10 features: ['Age', 'PS', 'AFP', 'ALT', 'AST', 'Albumin', 'Total_Bil', 'INR', 'Platelets', 'Creatinine']
Last 10 features: ['Gender_Male', 'Symptoms_Yes', 'Encephalopathy_Yes', 'Ascites_Yes', 'Alcohol_Yes', 'HBsAg_Yes', 'HCVAb_Yes', 'Cirrhosis_Yes', 'Diabetes_Yes', 'Smoking_Yes']
✅ Preprocessor saved successfully!
📁 Location: ../models/preprocessing/preprocessor.pkl
📊 Total features: 22
🔧 Preprocessor type: <class 'sklearn.compose._column_transformer.ColumnTransformer'>

🔍 Verifying saved preprocessor...
Type: <class 'dict'>
Keys: ['preprocessor', 'feature_names', 'original_features', 'categorical_features', 'numerical_features', 'feature_categories']
Feature names: 22
Preprocessor available: True
✅ Transformation successful!
   Input shape: (1, 22)
   Output shape: (1, 22)
   Expected features: 22
